# PCA in a broader context.
#
## PCA is basically the application of eigen-decomposition to data analysis.
## You can do supervised clustering, denoising, and regression.
## The main question to address is whether you use the class column or not.
## PCA is normally applied without the class column, while
## LDA, FA, and PLS use the class column.

In [ ]:
# install pre-requisites
!python -m pip install --upgrade pip
%pip install numpy pandas scikit-learn seaborn

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn import datasets
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

In [ ]:
# load some data

# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html
X, y = datasets.fetch_california_housing(return_X_y=True, as_frame=True)


In [ ]:
print('Number of missing values:', X.isna().sum().sum())

In [ ]:
model = Pipeline([
    ('scale', preprocessing.RobustScaler()),
    ('reg', LinearRegression())
])

# Show cross-validated R^2 scores
scores = cross_val_score(model, X, y, cv=10, n_jobs=-1)
print('R^2 scores for each fold:', scores)
print(f'Mean R^2: {scores.mean():.3f} (+/- {scores.std():.3f})')

In [ ]:
# we need to see the number of eigenvalues larger or equal 1.0

pcr = Pipeline([
    ('std', preprocessing.StandardScaler()),
    ('pca', PCA())
])

pcr.fit(X, y)
pca_step = pcr.named_steps['pca']

In [ ]:
ev = pca_step.explained_variance_        # eigenvalues
print("Eigenvalues:", ev)

explained_variance = pca_step.explained_variance_ratio_.cumsum()
print("Percentage of explained variance:", np.around(explained_variance*100, 2))

In [ ]:
# Create scree plot using matplotlib
plt.scatter(range(1, len(ev)+1), ev)
plt.plot(range(1, len(ev)+1), ev)
plt.title('Scree Plot')
plt.xlabel('Factors')
plt.ylabel('Eigenvalue')
plt.grid()
plt.show()

In [ ]:
# Option 1. let's now re-run our PCR with the n best components

pcr = Pipeline([
    ('std', preprocessing.StandardScaler()),
    ('pca', PCA(n_components = 5)),
    ('lin', LinearRegression())
])

# Show cross-validated R^2 scores
scores = cross_val_score(pcr, X, y, cv=10, n_jobs=-1)
print('R^2 scores for each fold:', scores)
print(f'Mean R^2: {scores.mean():.3f} (+/- {scores.std():.3f})')

In [ ]:
# Let's re-check the eigenvalues and explained variance ...
pcr.fit(X, y)
pca_step = pcr.named_steps['pca']
ev = pca_step.explained_variance_        # eigenvalues
explained_variance = pca_step.explained_variance_ratio_.sum()
print("PCR with ", len(ev), "Principal Components and explained variance of", round(explained_variance*100,2), "%")

In [ ]:
# Option 2. let's now re-run our PCR with 95% of explained variance
pcr.set_params(pca__n_components=0.95, pca__svd_solver='full')

In [ ]:
# Let's re-check the eigenvalues and explained variance ...
pcr.fit(X, y)
pca_step = pcr.named_steps['pca']
ev = pca_step.explained_variance_        # eigenvalues
explained_variance = pca_step.explained_variance_ratio_.sum()
print("PCR with ", len(ev), "Principal Components and explained variance of", round(explained_variance*100,2), "%")

In [ ]:
# Show cross-validated R^2 scores
scores = cross_val_score(pcr, X, y, cv=10, n_jobs=-1)
print('R^2 scores for each fold:', scores)
print(f'Mean R^2: {scores.mean():.3f} (+/- {scores.std():.3f})')

In [ ]:
# A related technique is Partial Least Square Regression.
# Similar to PCA, it transform the feature space into another
# dimension (principal components), but it uses the outcome variable.
# https://scikit-learn.org/stable/auto_examples/cross_decomposition/plot_compare_cross_decomposition.html

pls = Pipeline([
    ('pls', PLSRegression(n_components=5, scale=True))
])

# Show cross-validated R^2 scores
scores = cross_val_score(pls, X, y, cv=10, n_jobs=-1)
print('R^2 scores for each fold:', scores)
print(f'Mean R^2: {scores.mean():.3f} (+/- {scores.std():.3f})')

In [ ]:
# Let's try something else ...
from sklearn import ensemble

rfr = Pipeline([
    ('scale', preprocessing.PowerTransformer()),
    ('rf', ensemble.GradientBoostingRegressor())        # RF & GBM = 0.63
])

# Show cross-validated R^2 scores
scores = cross_val_score(rfr, X, y, cv=10, n_jobs=-1)
print('R^2 scores for each fold:', scores)
print(f'Mean R^2: {scores.mean():.3f} (+/- {scores.std():.3f})')